# Modelagem neural — MLP + Optuna e LSTM + Grid Search

Este notebook executa os quatro experimentos científicos definidos para o paper: MLP e LSTM sobre os conjuntos simples (16 atributos) e completo (36 atributos). O teste final permanece isolado durante toda a seleção de hiperparâmetros.

> **Custo computacional:** as configurações padrão abaixo executam o protocolo científico completo. Cada seção de treinamento salva seus artefatos imediatamente ao terminar.

## 1. Configuração e dependências

In [ ]:
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import optuna
import pandas as pd
import sklearn
import torch

from src.modeling.lstm_grid_search import (
    DEFAULT_LSTM_PARAM_GRID,
    DEFAULT_LSTM_PARAMS,
    DEFAULT_LSTM_SEEDS,
)
from src.modeling.lstm_training import (
    save_lstm_experiment_artifacts,
    train_lstm_experiment,
)
from src.modeling.mlp_optuna import (
    DEFAULT_MAX_EPOCHS,
    DEFAULT_MLP_SEEDS,
    DEFAULT_OPTUNA_TRIALS,
    DEFAULT_PRUNING_INTERVAL,
    DEFAULT_SAMPLER_SEED,
)
from src.modeling.mlp_training import (
    save_mlp_experiment_artifacts,
    train_mlp_experiment,
)
from src.modeling.modeling import (
    plot_model_comparison,
    prepare_modeling_artifact_directories,
)
from src.modeling.neural_experiments import (
    build_neural_experiment_matrix,
    prepare_neural_feature_sets,
)

OUTPUT_DIR = Path('results/modeling')
ARTIFACT_DIRS = prepare_modeling_artifact_directories(OUTPUT_DIR)

TEST_DAYS = 365
INITIAL_TRAIN_DAYS = 730
VALIDATION_DAYS = 180
STEP_DAYS = 180

SEEDS = tuple(DEFAULT_MLP_SEEDS)
DEVICE = None  # None seleciona CUDA quando disponível, senão CPU.

MLP_N_TRIALS = DEFAULT_OPTUNA_TRIALS
MLP_MAX_EPOCHS = DEFAULT_MAX_EPOCHS
MLP_PRUNING_INTERVAL = DEFAULT_PRUNING_INTERVAL
MLP_SAMPLER_SEED = DEFAULT_SAMPLER_SEED

LSTM_PARAM_GRID = {
    name: tuple(values)
    for name, values in DEFAULT_LSTM_PARAM_GRID.items()
}

neural_results = {}
artifact_paths = {}

print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'Optuna: {optuna.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'Device: {DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")}')

## 2. Preparação comum dos experimentos

Os dois conjuntos são alinhados por data e alvo. O holdout e os folds expansivos são compartilhados pelas quatro combinações.

In [ ]:
feature_sets = prepare_neural_feature_sets(
    test_days=TEST_DAYS,
    initial_train_days=INITIAL_TRAIN_DAYS,
    validation_days=VALIDATION_DAYS,
    step_days=STEP_DAYS,
)
experiments = build_neural_experiment_matrix(feature_sets=feature_sets)

dataset_summary = pd.DataFrame([
    {
        'Dataset': name,
        'Rows': len(data['full_dataset']),
        'DevelopmentRows': len(data['development_dataset']),
        'TestRows': len(data['test_dataset']),
        'Predictors': data['input_size'],
        'Folds': len(data['splits']),
    }
    for name, data in feature_sets.items()
])
display(dataset_summary)
display(feature_sets['simple']['holdout_summary'])
display(feature_sets['simple']['validation_summary'])

## 3. Persistência do protocolo experimental

Esta etapa registra divisões, versões, sementes e orçamentos antes dos treinamentos.

In [ ]:
feature_sets['simple']['holdout_summary'].to_csv(
    ARTIFACT_DIRS['validation'] / 'final_holdout.csv',
    index=False,
)
feature_sets['simple']['validation_summary'].to_csv(
    ARTIFACT_DIRS['validation'] / 'validation_windows.csv',
    index=False,
)

protocol = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'target': 'recurrences_lt_30m',
    'test_days': TEST_DAYS,
    'initial_train_days': INITIAL_TRAIN_DAYS,
    'validation_days': VALIDATION_DAYS,
    'step_days': STEP_DAYS,
    'seeds': list(SEEDS),
    'device_request': DEVICE,
    'feature_counts': {
        name: data['input_size']
        for name, data in feature_sets.items()
    },
    'mlp': {
        'method': 'Optuna TPE with median pruning',
        'n_trials': MLP_N_TRIALS,
        'max_epochs': MLP_MAX_EPOCHS,
        'pruning_interval': MLP_PRUNING_INTERVAL,
        'sampler_seed': MLP_SAMPLER_SEED,
        'selection_metric': 'mean expanding-window MAE',
    },
    'lstm': {
        'method': 'Grid Search',
        'fixed_parameters': DEFAULT_LSTM_PARAMS,
        'parameter_grid': {
            name: list(values)
            for name, values in LSTM_PARAM_GRID.items()
        },
        'selection_metric': 'mean expanding-window MAE',
    },
    'versions': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'optuna': optuna.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
    },
}
protocol_path = ARTIFACT_DIRS['metadata'] / 'neural_experiment_protocol.json'
protocol_path.write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False),
    encoding='utf-8',
)
print(f'Protocol saved to: {protocol_path}')

## 4. MLP + Optuna — conjunto simples

In [ ]:
mlp_simple = train_mlp_experiment(
    experiments['mlp_simple'],
    n_trials=MLP_N_TRIALS,
    seeds=SEEDS,
    max_epochs=MLP_MAX_EPOCHS,
    pruning_interval=MLP_PRUNING_INTERVAL,
    sampler_seed=MLP_SAMPLER_SEED,
    device=DEVICE,
)
neural_results['mlp_simple'] = mlp_simple
artifact_paths['mlp_simple'] = save_mlp_experiment_artifacts(
    mlp_simple,
    OUTPUT_DIR,
)
display(mlp_simple['trials_summary'].query("State == 'COMPLETE'").sort_values('Value').head(10))
display(mlp_simple['seed_metrics'])
display(pd.DataFrame([mlp_simple['test_metrics']]))

## 5. MLP + Optuna — conjunto completo

In [ ]:
mlp_complete = train_mlp_experiment(
    experiments['mlp_complete'],
    n_trials=MLP_N_TRIALS,
    seeds=SEEDS,
    max_epochs=MLP_MAX_EPOCHS,
    pruning_interval=MLP_PRUNING_INTERVAL,
    sampler_seed=MLP_SAMPLER_SEED,
    device=DEVICE,
)
neural_results['mlp_complete'] = mlp_complete
artifact_paths['mlp_complete'] = save_mlp_experiment_artifacts(
    mlp_complete,
    OUTPUT_DIR,
)
display(mlp_complete['trials_summary'].query("State == 'COMPLETE'").sort_values('Value').head(10))
display(mlp_complete['seed_metrics'])
display(pd.DataFrame([mlp_complete['test_metrics']]))

## 6. LSTM + Grid Search — conjunto simples

In [ ]:
lstm_simple = train_lstm_experiment(
    experiments['lstm_simple'],
    param_grid=LSTM_PARAM_GRID,
    seeds=SEEDS,
    device=DEVICE,
)
neural_results['lstm_simple'] = lstm_simple
artifact_paths['lstm_simple'] = save_lstm_experiment_artifacts(
    lstm_simple,
    OUTPUT_DIR,
)
display(lstm_simple['search_summary'])
display(lstm_simple['seed_metrics'])
display(pd.DataFrame([lstm_simple['test_metrics']]))

## 7. LSTM + Grid Search — conjunto completo

In [ ]:
lstm_complete = train_lstm_experiment(
    experiments['lstm_complete'],
    param_grid=LSTM_PARAM_GRID,
    seeds=SEEDS,
    device=DEVICE,
)
neural_results['lstm_complete'] = lstm_complete
artifact_paths['lstm_complete'] = save_lstm_experiment_artifacts(
    lstm_complete,
    OUTPUT_DIR,
)
display(lstm_complete['search_summary'])
display(lstm_complete['seed_metrics'])
display(pd.DataFrame([lstm_complete['test_metrics']]))

## 8. Comparação consolidada

A comparação é reconstruída a partir dos arquivos persistidos. Assim, ela também funciona quando as quatro seções são executadas em sessões diferentes.

In [ ]:
neural_metric_files = sorted(
    list(ARTIFACT_DIRS['metrics'].glob('mlp_*_test_metrics.json'))
    + list(ARTIFACT_DIRS['metrics'].glob('lstm_*_test_metrics.json'))
)
if not neural_metric_files:
    raise RuntimeError('Execute e salve ao menos um experimento neural antes da comparação')

persisted_neural_metrics = [
    json.loads(path.read_text(encoding='utf-8'))
    for path in neural_metric_files
]
neural_comparison = pd.DataFrame(persisted_neural_metrics)
neural_comparison['Coverage'] = 1.0
neural_comparison = neural_comparison.sort_values(['MAE', 'RMSE']).reset_index(drop=True)
neural_comparison.to_csv(
    ARTIFACT_DIRS['comparisons'] / 'neural_model_comparison.csv',
    index=False,
)
display(neural_comparison[[
    'Model', 'Dataset', 'Rows', 'MAE', 'RMSE', 'R2',
    'PoissonDeviance', 'OptimizationSeconds', 'RefitSeconds',
    'TotalExperimentSeconds',
]])

figure, _ = plot_model_comparison(neural_comparison, metric='MAE')
figure.savefig(
    ARTIFACT_DIRS['comparisons'] / 'neural_model_comparison.png',
    dpi=150,
    bbox_inches='tight',
)
plt.show()

## 9. Atualização da comparação geral

Os resultados neurais persistidos são combinados com Random Forest, XGBoost e baseline já presentes no projeto.

In [ ]:
comparison_path = ARTIFACT_DIRS['comparisons'] / 'model_comparison.csv'
if comparison_path.exists():
    previous_comparison = pd.read_csv(comparison_path)
    previous_comparison = previous_comparison.loc[
        ~previous_comparison['Model'].isin(['mlp', 'lstm'])
    ]
else:
    previous_comparison = pd.DataFrame()

comparison = pd.concat(
    [previous_comparison, neural_comparison],
    ignore_index=True,
    sort=False,
).sort_values(['MAE', 'RMSE']).reset_index(drop=True)
comparison.to_csv(comparison_path, index=False)
display(comparison[[
    'Model', 'Dataset', 'MAE', 'RMSE', 'R2',
    'PoissonDeviance', 'Coverage',
]])

figure, _ = plot_model_comparison(comparison, metric='MAE')
figure.savefig(
    ARTIFACT_DIRS['comparisons'] / 'model_comparison.png',
    dpi=150,
    bbox_inches='tight',
)
plt.show()

## 10. Manifesto dos artefatos neurais

In [ ]:
manifest_rows = []
for section, directory in ARTIFACT_DIRS.items():
    for path in sorted(directory.glob('*')):
        if path.is_file() and path.name.startswith(('mlp_', 'lstm_', 'neural_')):
            manifest_rows.append({
                'Section': section,
                'File': path.name,
                'Path': path.as_posix(),
                'Bytes': path.stat().st_size,
            })

artifact_manifest = pd.DataFrame(manifest_rows).sort_values(
    ['Section', 'File']
).reset_index(drop=True)
manifest_path = ARTIFACT_DIRS['comparisons'] / 'neural_artifact_manifest.csv'
artifact_manifest.to_csv(manifest_path, index=False)
display(artifact_manifest)
print(f'Artifact manifest saved to: {manifest_path}')

## Saídas

Cada treinamento persiste imediatamente checkpoint, metadados, métricas finais e por semente, previsões, resíduos e tabelas de validação. A comparação e o manifesto podem ser reconstruídos dos arquivos salvos mesmo após reiniciar o kernel.